# Model 3 — DINOv2 Vision Transformer on Leafsnap (Google Colab)

This notebook trains and evaluates a **DINOv2-pretrained ViT-Base/14 with four register tokens** on all 185 Leafsnap species. It is designed for Google Colab and saves durable artifacts to Google Drive.

## Development map

1. Configure Colab, GPU, Drive and reproducibility.
2. Copy `archive.zip` from Drive and extract it to fast local Colab storage.
3. Validate the official TSV manifest and use only the 30,866 original RGB images.
4. Load the exact leakage-aware 70/15/15 group split shared by Models 1 and 2, or create it identically on a first run.
5. Build DINOv2-compatible transforms, datasets and data loaders.
6. Train the classifier head for 5 epochs while the transformer is frozen.
7. Unfreeze the full transformer and fine-tune for up to 25 epochs at a lower learning rate.
8. Resume safely after a Colab interruption using phase-aware checkpoints.
9. Evaluate Top-1, Top-5, macro/weighted F1 and lab-versus-field performance.
10. Save predictions, metrics, diagnostics and provide single-image inference.

### Dataset assumptions

- Drive archive: `MyDrive/Major Project/archive.zip`
- Expected records: 30,866 (23,147 lab + 7,719 field)
- Expected classes: 185 overall; field observations cover 184 classes
- `dataset/segmented` contains masks, not extra training images
- Lab views from the same likely specimen are kept in one split to reduce leakage
- Existing ConvNeXt/EfficientNet results are not directly comparable unless those models are rerun on this same saved split and preprocessing

> Run the notebook from top to bottom only after the Drive upload has completed. Training results are intentionally not pre-filled; they must come from the executed run.


## 1. Setup

Use **Runtime → Change runtime type → GPU** before running. `timm` supplies the pretrained DINOv2 Vision Transformer. The first model-creation run downloads the pretrained checkpoint, so Colab must have internet access.


In [ ]:
%pip install -q -U timm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import hashlib
import json
import os
import random
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import timm
import torch
import torch.nn as nn
from PIL import Image
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# -------------------------
# User-editable configuration
# -------------------------
DRIVE_PROJECT = Path('/content/drive/MyDrive/Major Project')
ARCHIVE_PATH = DRIVE_PROJECT / 'archive.zip'
LOCAL_ARCHIVE = Path('/content/archive.zip')
EXTRACT_ROOT = Path('/content/leafsnap_data')
SPLIT_PATH = DRIVE_PROJECT / 'splits/leafsnap_group_split_seed42.csv'
SPLIT_META_PATH = DRIVE_PROJECT / 'splits/leafsnap_group_split_seed42.json'
RUN_DIR = DRIVE_PROJECT / 'checkpoints/vit_b14_dinov2'
RESULTS_DIR = DRIVE_PROJECT / 'results/vit_b14_dinov2'

SEED = 42
IMAGE_SIZE = 224
MODEL_NAME = 'vit_base_patch14_reg4_dinov2.lvd142m'
BATCH_SIZE = 16             # Physical batch; reduce to 8 only if CUDA runs out of memory
ACCUMULATION_STEPS = 2      # Effective batch size = 32 by default
NUM_WORKERS = 2
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 25
TOTAL_EPOCHS = HEAD_EPOCHS + FINETUNE_EPOCHS
HEAD_LEARNING_RATE = 1e-3
FINETUNE_LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 6
RESUME = True
FORCE_RECREATE_SPLIT = False

for directory in [SPLIT_PATH.parent, RUN_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('GPU not detected. Select Runtime > Change runtime type > GPU, then rerun.')

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.titlesize': 13, 'axes.labelsize': 11})

print('Device:', device, '-', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| timm:', timm.__version__, '| scikit-learn:', sklearn.__version__)
print('Drive archive:', ARCHIVE_PATH)

## 2. Stage and validate the dataset

The archive remains permanent in Drive. It is copied and extracted under `/content` because training directly from thousands of Drive files is slow.

In [ ]:
assert ARCHIVE_PATH.exists(), f'Archive not found: {ARCHIVE_PATH}. Wait for the Drive upload to finish.'

drive_size = ARCHIVE_PATH.stat().st_size
if not LOCAL_ARCHIVE.exists() or LOCAL_ARCHIVE.stat().st_size != drive_size:
    print(f'Copying {drive_size / 2**30:.2f} GiB from Drive to Colab...')
    shutil.copy2(ARCHIVE_PATH, LOCAL_ARCHIVE)
else:
    print('Local archive already matches the Drive archive.')

extract_marker = EXTRACT_ROOT / '.extraction_complete'
if not extract_marker.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting archive...')
    shutil.unpack_archive(LOCAL_ARCHIVE, EXTRACT_ROOT)
    extract_marker.write_text('complete', encoding='utf-8')
else:
    print('Existing extraction detected.')

manifest_candidates = list(EXTRACT_ROOT.rglob('leafsnap-dataset-images.txt'))
assert len(manifest_candidates) == 1, (
    f'Expected exactly one manifest, found {len(manifest_candidates)}: {manifest_candidates}'
)
MANIFEST_PATH = manifest_candidates[0]

def locate_dataset_root(manifest_path: Path) -> Path:
    candidates = [manifest_path.parent / 'leafsnap-dataset', manifest_path.parent, *manifest_path.parents]
    for candidate in candidates:
        if (candidate / 'dataset/images/lab').exists() and (candidate / 'dataset/images/field').exists():
            return candidate
    raise FileNotFoundError('Could not locate leafsnap-dataset/dataset/images after extraction.')

DATASET_ROOT = locate_dataset_root(MANIFEST_PATH)
IMAGES_ROOT = DATASET_ROOT / 'dataset/images'
MASKS_ROOT = DATASET_ROOT / 'dataset/segmented'
manifest_sha256 = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
metadata = pd.read_csv(MANIFEST_PATH, sep='\t')

required_columns = {'file_id', 'image_path', 'segmented_path', 'species', 'source'}
assert required_columns.issubset(metadata.columns), f'Missing columns: {required_columns - set(metadata.columns)}'
assert len(metadata) == 30_866, f'Expected 30,866 records, found {len(metadata):,}'
assert metadata['file_id'].is_unique and metadata['image_path'].is_unique
assert metadata['species'].nunique() == 185
assert metadata['source'].value_counts().to_dict() == {'lab': 23_147, 'field': 7_719}

missing_originals = [p for p in metadata['image_path'] if not (DATASET_ROOT / p).is_file()]
assert not missing_originals, f'Missing original images, first examples: {missing_originals[:5]}'

print('Manifest:', MANIFEST_PATH)
print('Dataset root:', DATASET_ROOT)
print(f'Validated {len(metadata):,} original images across {metadata.species.nunique()} species.')
print(metadata['source'].value_counts().rename('images'))

## 3. Create or reuse the shared split

The split is created **within each species** so every class appears in train, validation and test. Lab filenames commonly end in four view numbers; those views are grouped. Obvious field sequences are also grouped conservatively. The saved CSV and manifest fingerprint make the exact partition reusable by Models 2–4.

In [ ]:
def derive_group_id(row: pd.Series) -> str:
    stem = Path(row['image_path']).stem
    if row['source'] == 'lab':
        group_stem = re.sub(r'-\d+$', '', stem)
    elif re.fullmatch(r'.+_\d+', stem):
        group_stem = re.sub(r'_\d+$', '', stem)
    elif re.fullmatch(r'\d{14}', stem):
        group_stem = stem[:10]
    else:
        group_stem = stem
    return f"{row['source']}/{row['species']}/{group_stem}"

def build_group_split(frame: pd.DataFrame, seed: int) -> pd.DataFrame:
    split_frame = frame.copy()
    split_frame['group_id'] = split_frame.apply(derive_group_id, axis=1)
    split_frame['split'] = ''
    classes = sorted(split_frame['species'].unique())

    for class_index, species in enumerate(classes):
        species_mask = split_frame['species'].eq(species)
        groups = split_frame.loc[species_mask, 'group_id'].drop_duplicates().to_numpy()
        if len(groups) < 3:
            raise ValueError(f'{species} has only {len(groups)} groups; at least 3 are required.')

        rng = np.random.default_rng(seed + class_index)
        groups = rng.permutation(groups)
        n_train = max(1, int(round(0.70 * len(groups))))
        n_val = max(1, int(round(0.15 * len(groups))))
        if n_train + n_val >= len(groups):
            n_train, n_val = len(groups) - 2, 1

        assignment = {group: 'train' for group in groups[:n_train]}
        assignment.update({group: 'val' for group in groups[n_train:n_train + n_val]})
        assignment.update({group: 'test' for group in groups[n_train + n_val:]})
        split_frame.loc[species_mask, 'split'] = split_frame.loc[species_mask, 'group_id'].map(assignment)

    class_to_index = {name: index for index, name in enumerate(classes)}
    split_frame['class_index'] = split_frame['species'].map(class_to_index).astype(int)
    return split_frame

if SPLIT_PATH.exists() and SPLIT_META_PATH.exists() and not FORCE_RECREATE_SPLIT:
    split_metadata = json.loads(SPLIT_META_PATH.read_text(encoding='utf-8'))
    if split_metadata.get('manifest_sha256') != manifest_sha256:
        raise RuntimeError('Saved split belongs to a different manifest. Set FORCE_RECREATE_SPLIT=True after review.')
    if split_metadata.get('seed') != SEED:
        raise RuntimeError('Saved split uses a different seed. Set FORCE_RECREATE_SPLIT=True after review.')
    split_df = pd.read_csv(SPLIT_PATH)
    print('Loaded existing shared split:', SPLIT_PATH)
else:
    split_df = build_group_split(metadata, SEED)
    split_df.to_csv(SPLIT_PATH, index=False)
    split_metadata = {
        'seed': SEED, 'ratios_by_group': {'train': 0.70, 'val': 0.15, 'test': 0.15},
        'manifest_sha256': manifest_sha256, 'records': int(len(split_df)),
        'classes': int(split_df['species'].nunique()),
        'grouping': 'lab strip final -view; field strip _sequence or 14-digit timestamp suffix'
    }
    SPLIT_META_PATH.write_text(json.dumps(split_metadata, indent=2), encoding='utf-8')
    print('Created shared split:', SPLIT_PATH)

# Split integrity checks
assert len(split_df) == len(metadata)
assert set(split_df['file_id']) == set(metadata['file_id'])
assert split_df['split'].isin(['train', 'val', 'test']).all()
assert split_df.groupby('group_id')['split'].nunique().max() == 1, 'Group leakage detected.'
for split_name in ['train', 'val', 'test']:
    assert split_df.loc[split_df['split'].eq(split_name), 'species'].nunique() == 185

split_summary = pd.crosstab(split_df['split'], split_df['source']).reindex(['train', 'val', 'test'])
split_summary['total'] = split_summary.sum(axis=1)
split_summary['share'] = split_summary['total'] / len(split_df)
display(split_summary)

ax = split_summary[['lab', 'field']].plot(
    kind='bar', stacked=True, color=['#35618f', '#d89b2b'], edgecolor='#333333'
)
ax.set_title('Leafsnap images by split and source')
ax.set_xlabel('Split')
ax.set_ylabel('Images')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Source')
plt.tight_layout()
plt.show()

## 4. Data pipeline

Transforms preserve the entire leaf instead of using aggressive random crops, which could remove small or narrow leaves. DINOv2 uses bicubic resizing and standard ImageNet mean/std normalization. The 224-pixel input is divisible by the 14-pixel patch size and keeps the experiment practical on a Colab T4.


In [ ]:
DINO_MEAN = (0.485, 0.456, 0.406)
DINO_STD = (0.229, 0.224, 0.225)
PATCH_SIZE = 14
assert IMAGE_SIZE % PATCH_SIZE == 0, 'IMAGE_SIZE must be divisible by the 14-pixel patch size.'

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.25),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.90, 1.10), fill=255),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(DINO_MEAN, DINO_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(DINO_MEAN, DINO_STD),
])

CLASS_NAMES = sorted(split_df['species'].unique())
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

class LeafsnapDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, dataset_root: Path, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.dataset_root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = self.dataset_root / row['image_path']
        with Image.open(image_path) as image_file:
            image = image_file.convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['class_index']), index

train_df = split_df[split_df['split'].eq('train')].reset_index(drop=True)
val_df = split_df[split_df['split'].eq('val')].reset_index(drop=True)
test_df = split_df[split_df['split'].eq('test')].reset_index(drop=True)
train_dataset = LeafsnapDataset(train_df, DATASET_ROOT, train_transform)
val_dataset = LeafsnapDataset(val_df, DATASET_ROOT, eval_transform)
test_dataset = LeafsnapDataset(test_df, DATASET_ROOT, eval_transform)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)
loader_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=False, worker_init_fn=seed_worker
)
train_loader = DataLoader(train_dataset, shuffle=True, generator=loader_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')
print(f'Physical batch: {BATCH_SIZE} | Effective training batch: {BATCH_SIZE * ACCUMULATION_STEPS}')
images, labels, row_indices = next(iter(train_loader))
print('Batch images:', tuple(images.shape), '| labels:', tuple(labels.shape), '| range:', (images.min().item(), images.max().item()))


In [ ]:
def denormalize(tensor):
    mean = torch.tensor(DINO_MEAN).view(3, 1, 1)
    std = torch.tensor(DINO_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for axis, image_tensor, label in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(denormalize(image_tensor).permute(1, 2, 0))
    axis.set_title(CLASS_NAMES[int(label)], fontsize=9)
    axis.axis('off')
fig.suptitle('Augmented training examples', y=1.01)
plt.tight_layout()
plt.show()

## 5. DINOv2 Vision Transformer and training utilities

The project guide labels this comparison “ViT-B/16, DINOv2-pretrained,” but its recommended timm checkpoint is `vit_base_patch14_reg4_dinov2.lvd142m`: a ViT-Base with 14×14 patches and four register tokens. This notebook follows that exact recommended checkpoint.

The published checkpoint is configured for 518×518 input. Passing `img_size=224` makes timm resample the positional embeddings while greatly reducing Colab memory and training time. Fine-tuning uses the guide's two-phase recipe: a 5-epoch head-only warmup at `1e-3`, followed by up to 25 full-model epochs at `3e-5`, AdamW weight decay `0.05`, label smoothing `0.1`, cosine decay, mixed precision and gradient accumulation.


In [ ]:
model = timm.create_model(
    MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, img_size=IMAGE_SIZE
).to(device)
assert hasattr(model, 'head'), 'Expected a timm Vision Transformer with a classification head.'
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f'DINOv2 ViT parameters: {total_parameters:,} total')
print('Pretrained input configuration:', model.pretrained_cfg.get('input_size'))
print('Experiment input size:', (3, IMAGE_SIZE, IMAGE_SIZE))

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
USE_AMP = device.type == 'cuda'
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

def phase_for_epoch(epoch):
    return 'head' if epoch < HEAD_EPOCHS else 'finetune'

def configure_trainable_parameters(phase):
    if phase == 'head':
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.head.parameters():
            parameter.requires_grad = True
    elif phase == 'finetune':
        for parameter in model.parameters():
            parameter.requires_grad = True
    else:
        raise ValueError(f'Unknown phase: {phase}')
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f'Phase={phase} | trainable parameters={trainable:,}/{total_parameters:,}')

def build_optimizer_and_scheduler(phase):
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    learning_rate = HEAD_LEARNING_RATE if phase == 'head' else FINETUNE_LEARNING_RATE
    optimizer = torch.optim.AdamW(trainable_parameters, lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = None
    if phase == 'finetune':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6
        )
    return optimizer, scheduler

def atomic_torch_save(payload, destination: Path) -> None:
    """Write locally first, then replace the Drive file to reduce partial-checkpoint risk."""
    local_temporary = Path('/content') / f'.{destination.name}.local.tmp'
    drive_temporary = destination.with_suffix(destination.suffix + '.tmp')
    torch.save(payload, local_temporary)
    shutil.copy2(local_temporary, drive_temporary)
    os.replace(drive_temporary, destination)
    local_temporary.unlink(missing_ok=True)

def capture_rng_state():
    return {
        'python': random.getstate(), 'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(), 'cuda': torch.cuda.get_rng_state_all(),
        'loader_generator': loader_generator.get_state(),
    }

def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])
    loader_generator.set_state(state['loader_generator'])

def top_k_correct(logits, targets, k=5):
    k = min(k, logits.shape[1])
    top_indices = logits.topk(k, dim=1).indices
    return top_indices.eq(targets.view(-1, 1)).any(dim=1).sum().item()

def run_training_epoch(model, loader, optimizer, phase):
    if phase == 'head':
        model.eval()
        model.head.train()
    else:
        model.train()

    running_loss = running_top1 = running_top5 = sample_count = 0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(enumerate(loader), total=len(loader), desc=f'train/{phase}', leave=False)

    for step, (batch_images, batch_targets, _) in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)
        window_start = (step // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
        window_size = min(ACCUMULATION_STEPS, len(loader) - window_start)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(batch_images)
            loss = criterion(logits, batch_targets)
            scaled_loss = loss / window_size
        scaler.scale(scaled_loss).backward()

        should_step = ((step + 1) % ACCUMULATION_STEPS == 0) or (step + 1 == len(loader))
        if should_step:
            scaler.unscale_(optimizer)
            trainable_parameters = [
                parameter for parameter in model.parameters() if parameter.requires_grad
            ]
            torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = batch_targets.size(0)
        running_loss += loss.item() * batch_size
        running_top1 += logits.argmax(1).eq(batch_targets).sum().item()
        running_top5 += top_k_correct(logits, batch_targets, 5)
        sample_count += batch_size
        progress.set_postfix(loss=f'{running_loss / sample_count:.4f}')

    return {
        'loss': running_loss / sample_count, 'top1': running_top1 / sample_count,
        'top5': running_top5 / sample_count
    }

@torch.inference_mode()
def evaluate_loader(model, loader):
    model.eval()
    running_loss = running_top1 = running_top5 = sample_count = 0
    all_targets, all_predictions = [], []
    for batch_images, batch_targets, _ in tqdm(loader, desc='evaluate', leave=False):
        batch_images = batch_images.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(batch_images)
            loss = criterion(logits, batch_targets)
        predictions = logits.argmax(1)
        batch_size = batch_targets.size(0)
        running_loss += loss.item() * batch_size
        running_top1 += predictions.eq(batch_targets).sum().item()
        running_top5 += top_k_correct(logits, batch_targets, 5)
        sample_count += batch_size
        all_targets.extend(batch_targets.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
    return {
        'loss': running_loss / sample_count, 'top1': running_top1 / sample_count,
        'top5': running_top5 / sample_count,
        'macro_f1': f1_score(all_targets, all_predictions, average='macro', zero_division=0)
    }


## 6. Two-phase training with checkpointing and resume

`last.pt` stores the model, optimizer, scheduler, scaler, RNG and active-phase state after every epoch. `best.pt` stores the model with the highest validation Top-1 accuracy across both phases. Resuming restores the correct phase automatically. Set `RESUME=False` only when intentionally starting a new run.


In [ ]:
LAST_CHECKPOINT = RUN_DIR / 'last.pt'
BEST_CHECKPOINT = RUN_DIR / 'best.pt'
HISTORY_PATH = RESULTS_DIR / 'training_history.csv'

run_config = {
    'model': MODEL_NAME, 'pretrained': True, 'num_classes': NUM_CLASSES,
    'image_size': IMAGE_SIZE, 'native_pretrained_image_size': 518, 'patch_size': PATCH_SIZE,
    'physical_batch_size': BATCH_SIZE,
    'accumulation_steps': ACCUMULATION_STEPS,
    'effective_batch_size': BATCH_SIZE * ACCUMULATION_STEPS,
    'head_epochs': HEAD_EPOCHS, 'finetune_epochs': FINETUNE_EPOCHS,
    'head_learning_rate': HEAD_LEARNING_RATE,
    'finetune_learning_rate': FINETUNE_LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING,
    'seed': SEED, 'manifest_sha256': manifest_sha256, 'split_path': str(SPLIT_PATH)
}
(RESULTS_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

start_epoch, best_val_top1, epochs_without_improvement = 0, -1.0, 0
history = []

if RESUME and LAST_CHECKPOINT.exists():
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    if checkpoint['manifest_sha256'] != manifest_sha256 or checkpoint['class_names'] != CLASS_NAMES:
        raise RuntimeError('Checkpoint dataset/classes do not match this run.')
    saved_config = checkpoint.get('config', {})
    if saved_config.get('model') != MODEL_NAME or saved_config.get('image_size') != IMAGE_SIZE:
        raise RuntimeError('Checkpoint model/input configuration does not match this notebook.')
    model.load_state_dict(checkpoint['model_state'])
    active_phase = checkpoint['phase']
    configure_trainable_parameters(active_phase)
    optimizer, scheduler = build_optimizer_and_scheduler(active_phase)
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    if scheduler is not None and checkpoint.get('scheduler_state') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_top1 = checkpoint['best_val_top1']
    epochs_without_improvement = checkpoint['epochs_without_improvement']
    history = checkpoint['history']
    if 'rng_state' in checkpoint:
        restore_rng_state(checkpoint['rng_state'])
    print(
        f"Resuming at epoch {start_epoch + 1}/{TOTAL_EPOCHS}; "
        f"saved phase={active_phase}; best validation Top-1={best_val_top1:.4f}"
    )
else:
    active_phase = phase_for_epoch(0)
    configure_trainable_parameters(active_phase)
    optimizer, scheduler = build_optimizer_and_scheduler(active_phase)

training_started = time.time()
for epoch in range(start_epoch, TOTAL_EPOCHS):
    desired_phase = phase_for_epoch(epoch)
    if desired_phase != active_phase:
        active_phase = desired_phase
        configure_trainable_parameters(active_phase)
        optimizer, scheduler = build_optimizer_and_scheduler(active_phase)
        epochs_without_improvement = 0
        print('Started full-model fine-tuning with a fresh low-learning-rate optimizer.')

    epoch_started = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    train_metrics = run_training_epoch(model, train_loader, optimizer, active_phase)
    val_metrics = evaluate_loader(model, val_loader)
    if scheduler is not None:
        scheduler.step()

    epoch_record = {
        'epoch': epoch + 1, 'phase': active_phase, 'lr': current_lr,
        'train_loss': train_metrics['loss'], 'train_top1': train_metrics['top1'],
        'train_top5': train_metrics['top5'], 'val_loss': val_metrics['loss'],
        'val_top1': val_metrics['top1'], 'val_top5': val_metrics['top5'],
        'val_macro_f1': val_metrics['macro_f1'],
        'minutes': (time.time() - epoch_started) / 60
    }
    history.append(epoch_record)

    improved = val_metrics['top1'] > best_val_top1
    if improved:
        best_val_top1 = val_metrics['top1']
        epochs_without_improvement = 0
        atomic_torch_save({
            'epoch': epoch, 'phase': active_phase, 'model_state': model.state_dict(),
            'class_names': CLASS_NAMES, 'manifest_sha256': manifest_sha256,
            'config': run_config, 'val_metrics': val_metrics
        }, BEST_CHECKPOINT)
    elif active_phase == 'finetune':
        epochs_without_improvement += 1

    atomic_torch_save({
        'epoch': epoch, 'phase': active_phase, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state': scaler.state_dict(), 'best_val_top1': best_val_top1,
        'epochs_without_improvement': epochs_without_improvement, 'history': history,
        'class_names': CLASS_NAMES, 'manifest_sha256': manifest_sha256,
        'config': run_config, 'rng_state': capture_rng_state()
    }, LAST_CHECKPOINT)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch + 1:02d}/{TOTAL_EPOCHS} [{active_phase}] | "
        f"train loss {train_metrics['loss']:.4f}, top1 {train_metrics['top1']:.4f} | "
        f"val loss {val_metrics['loss']:.4f}, top1 {val_metrics['top1']:.4f}, "
        f"top5 {val_metrics['top5']:.4f}, macro-F1 {val_metrics['macro_f1']:.4f}"
    )
    if (
        active_phase == 'finetune'
        and epochs_without_improvement >= EARLY_STOPPING_PATIENCE
    ):
        print(f'Early stopping after {EARLY_STOPPING_PATIENCE} fine-tuning epochs without improvement.')
        break

print(f'Training cell completed in {(time.time() - training_started) / 60:.1f} minutes.')
print('Best checkpoint:', BEST_CHECKPOINT)


In [ ]:
history_df = pd.DataFrame(history)
assert not history_df.empty, 'No training history found.'
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Train', color='#35618f')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Validation', color='#d89b2b')
axes[0].axvline(HEAD_EPOCHS + 0.5, color='#666666', linestyle=':', label='Full fine-tuning starts')
axes[0].set(title='Cross-entropy loss by epoch', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['train_top1'], marker='o', label='Train Top-1', color='#35618f')
axes[1].plot(history_df['epoch'], history_df['val_top1'], marker='o', label='Validation Top-1', color='#d89b2b')
axes[1].plot(history_df['epoch'], history_df['val_macro_f1'], linestyle='--', label='Validation macro-F1', color='#8a5a9e')
axes[1].axvline(HEAD_EPOCHS + 0.5, color='#666666', linestyle=':')
axes[1].set(title='Classification performance by epoch', xlabel='Epoch', ylabel='Score', ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
figure_path = RESULTS_DIR / 'training_curves.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
display(history_df.tail(10))
print('Saved:', figure_path)


## 7. Test evaluation

The best validation checkpoint is evaluated exactly once on the held-out test split. Overall results are supplemented with lab and field results because testing whether transformer features reduce the source-domain gap is a central Model 3 question. Field metrics cover the 184 species observed in field data; the classifier still has 185 output classes.


In [ ]:
assert BEST_CHECKPOINT.exists(), 'Best checkpoint not found; run the training cell first.'
best_checkpoint = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state'])
model.eval()

@torch.inference_mode()
def collect_predictions(model, loader, dataset):
    records = []
    for batch_images, batch_targets, batch_indices in tqdm(loader, desc='test predictions'):
        batch_images = batch_images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            probabilities = model(batch_images).softmax(dim=1)
        top_probabilities, top_indices = probabilities.topk(5, dim=1)
        for target, row_index, indices, probs in zip(
            batch_targets.tolist(), batch_indices.tolist(), top_indices.cpu().tolist(), top_probabilities.cpu().tolist()
        ):
            row = dataset.frame.iloc[row_index]
            records.append({
                'file_id': row['file_id'], 'image_path': row['image_path'],
                'source': row['source'], 'true_species': CLASS_NAMES[target],
                'predicted_species': CLASS_NAMES[indices[0]],
                'correct_top1': int(indices[0] == target),
                'correct_top5': int(target in indices),
                'top5_species': '|'.join(CLASS_NAMES[index] for index in indices),
                'top5_probabilities': '|'.join(f'{probability:.6f}' for probability in probs),
                'true_index': target, 'predicted_index': indices[0]
            })
    return pd.DataFrame(records)

predictions_df = collect_predictions(model, test_loader, test_dataset)
predictions_path = RESULTS_DIR / 'test_predictions.csv'
predictions_df.to_csv(predictions_path, index=False)

def calculate_metrics(frame):
    y_true = frame['true_index'].to_numpy()
    y_pred = frame['predicted_index'].to_numpy()
    return {
        'images': int(len(frame)), 'observed_classes': int(frame['true_species'].nunique()),
        'top1_accuracy': float(frame['correct_top1'].mean()),
        'top5_accuracy': float(frame['correct_top5'].mean()),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }

metrics = {'overall': calculate_metrics(predictions_df)}
for source_name in ['lab', 'field']:
    metrics[source_name] = calculate_metrics(predictions_df[predictions_df['source'].eq(source_name)])
metrics_path = RESULTS_DIR / 'test_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
domain_gap = {
    'top1_accuracy_lab_minus_field': metrics['lab']['top1_accuracy'] - metrics['field']['top1_accuracy'],
    'macro_f1_lab_minus_field': metrics['lab']['macro_f1'] - metrics['field']['macro_f1'],
}
domain_gap_path = RESULTS_DIR / 'domain_gap.json'
domain_gap_path.write_text(json.dumps(domain_gap, indent=2), encoding='utf-8')
metrics_table = pd.DataFrame(metrics).T
display(metrics_table.style.format({
    'top1_accuracy': '{:.4f}', 'top5_accuracy': '{:.4f}', 'macro_f1': '{:.4f}',
    'weighted_f1': '{:.4f}', 'balanced_accuracy': '{:.4f}'
}))
print('Saved predictions:', predictions_path)
print('Saved metrics:', metrics_path)
print('Lab minus field gap:', domain_gap)
print('Saved domain gap:', domain_gap_path)

In [ ]:
y_true = predictions_df['true_index'].to_numpy()
y_pred = predictions_df['predicted_index'].to_numpy()
report = classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
per_class_df = pd.DataFrame(report).T.loc[CLASS_NAMES].reset_index(names='species')
per_class_path = RESULTS_DIR / 'per_class_metrics.csv'
per_class_df.to_csv(per_class_path, index=False)

confusion = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
flat_order = np.argsort(off_diagonal.ravel())[::-1]
confusion_rows = []
for flat_index in flat_order:
    true_index, predicted_index = np.unravel_index(flat_index, off_diagonal.shape)
    count = int(off_diagonal[true_index, predicted_index])
    if count == 0 or len(confusion_rows) == 20:
        break
    confusion_rows.append({
        'pair': f'{CLASS_NAMES[true_index]} → {CLASS_NAMES[predicted_index]}', 'images': count
    })
confusion_pairs_df = pd.DataFrame(confusion_rows)
confusion_pairs_df.to_csv(RESULTS_DIR / 'top_confusion_pairs.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(19, 7))
metric_names = ['top1_accuracy', 'top5_accuracy', 'macro_f1', 'weighted_f1']
metric_labels = ['Top-1', 'Top-5', 'Macro F1', 'Weighted F1']
axes[0].bar(metric_labels, [metrics['overall'][name] for name in metric_names], color='#35618f')
axes[0].set(title='Overall held-out test metrics', ylabel='Score', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=30)
for index, value in enumerate([metrics['overall'][name] for name in metric_names]):
    axes[0].text(index, value + 0.02, f'{value:.3f}', ha='center')

worst_classes = per_class_df.nsmallest(20, 'f1-score').sort_values('f1-score')
axes[1].barh(worst_classes['species'], worst_classes['f1-score'], color='#d89b2b')
axes[1].set(title='Lowest 20 per-class F1 scores', xlabel='F1 score', xlim=(0, 1))

if not confusion_pairs_df.empty:
    plot_pairs = confusion_pairs_df.sort_values('images')
    axes[2].barh(plot_pairs['pair'], plot_pairs['images'], color='#8a5a9e')
axes[2].set(title='Most frequent Top-1 confusion pairs', xlabel='Test images')
plt.tight_layout()
diagnostics_path = RESULTS_DIR / 'test_diagnostics.png'
plt.savefig(diagnostics_path, dpi=160, bbox_inches='tight')
plt.show()
display(confusion_pairs_df)
print('Saved per-class metrics:', per_class_path)
print('Saved diagnostics:', diagnostics_path)

## 8. Single-image inference

Pass any JPG/PNG path from Colab or mounted Drive to `predict_image`. The function uses the same DINOv2 evaluation preprocessing as the test set and returns the five most likely species.


In [ ]:
@torch.inference_mode()
def predict_image(image_path, top_k=5):
    image_path = Path(image_path)
    with Image.open(image_path) as image_file:
        image = image_file.convert('RGB')
    input_tensor = eval_transform(image).unsqueeze(0).to(device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        probabilities = model(input_tensor).softmax(dim=1)[0]
    top_probabilities, top_indices = probabilities.topk(min(top_k, NUM_CLASSES))
    result = pd.DataFrame({
        'species': [CLASS_NAMES[index] for index in top_indices.cpu().tolist()],
        'probability': top_probabilities.cpu().tolist(),
    })
    display(image.resize((min(600, image.width), int(image.height * min(600, image.width) / image.width))))
    display(result.style.format({'probability': '{:.2%}'}))
    return result

example_row = test_df.sample(1, random_state=SEED).iloc[0]
print('Expected species:', example_row['species'], '| source:', example_row['source'])
_ = predict_image(DATASET_ROOT / example_row['image_path'])

## 9. Completion checks and saved outputs

A successful run should produce these permanent Drive artifacts:

- `splits/leafsnap_group_split_seed42.csv` and its JSON fingerprint
- `checkpoints/vit_b14_dinov2/last.pt` for phase-aware resuming
- `checkpoints/vit_b14_dinov2/best.pt` for final inference
- `results/vit_b14_dinov2/run_config.json`
- `results/vit_b14_dinov2/training_history.csv` and `training_curves.png`
- `results/vit_b14_dinov2/test_metrics.json` and `domain_gap.json`
- `results/vit_b14_dinov2/test_predictions.csv`
- `results/vit_b14_dinov2/per_class_metrics.csv`
- `results/vit_b14_dinov2/top_confusion_pairs.csv` and `test_diagnostics.png`

Do not change the saved split for Model 4. It must load this same CSV so architecture comparisons use identical images.
